# Module 6 · Enhancing Data with Wikidata *(optional)*

*Jupyter Book, enrichment notebook.*

In Module 2 the *same* artist hid under many name strings. This notebook applies the **durable fix**:
attach a stable **Wikidata identifier (Q-ID)** to each entity in your dataset, and pull one **feature**
from Wikidata back into your table. Two records that share a Q-ID are the same thing, however their names
are spelled, that is **authority control**.

**We enhance the `institution` column** of the Van Gogh dataset: reconcile each holding institution to its
Wikidata item and add its **country** (Wikidata property *P17*).

> **No API key needed**, Wikidata is fully open. You do need an **internet connection**. If you're
> offline (or a firewall blocks Wikidata), the notebook still runs and explains what would happen; run it
> again with internet to produce the real result.


## 0 · Configuration


In [14]:
# Which file to enhance, which column holds the entity, and what to add.
INPUT_FILE   = "van_gogh_filtered.csv"   # produced in Module 4 (falls back to combined/sample)
ENTITY_COLUMN = "institution"            # the column to reconcile (try a person/place column in your data)
FEATURE       = "country"                # the Wikidata feature to add (this notebook implements country = P17)
LANG          = "en"                     # language for labels
SAMPLE_ONLY   = False                    # True = only reconcile the first few unique entities (quick test)

print(f"Enhancing '{ENTITY_COLUMN}' in {INPUT_FILE}; adding: {FEATURE}")

Enhancing 'institution' in van_gogh_filtered.csv; adding: country


In [15]:
from pathlib import Path
import pandas as pd
import requests

def find_dataset(name):
    for base in [Path("datasets"), Path("../datasets")]:
        if (base / name).exists():
            return base / name
    raise FileNotFoundError(name)

def datasets_dir():
    for base in [Path("datasets"), Path("../datasets")]:
        if base.exists():
            return base
    Path("datasets").mkdir(parents=True, exist_ok=True); return Path("datasets")

## 1 · Load the data


In [16]:
def load_input():
    for name in [INPUT_FILE, "van_gogh_filtered.csv", "van_gogh_combined.csv", "van_gogh_europeana.csv"]:
        try:
            p = find_dataset(name); print("Loaded", p); return pd.read_csv(p)
        except FileNotFoundError:
            continue
    raise FileNotFoundError("No dataset found, run Modules 2 and 4 first.")

df = load_input()
entities = sorted(df[ENTITY_COLUMN].dropna().unique())
if SAMPLE_ONLY:
    entities = entities[:3]
print(f"{len(entities)} unique '{ENTITY_COLUMN}' values to reconcile.")
entities

Loaded ../datasets/van_gogh_filtered.csv
10 unique 'institution' values to reconcile.


['Austrian Gallery Belvedere',
 'Catholic University of Leuven',
 'Digital Library for Dutch Literature',
 'Digital Memory of Catalonia',
 'Finnish National Gallery',
 'International Institute of Social History',
 'Nationalmuseum Sweden',
 'Swedish Air Force Museum',
 'The Israel Museum, Jerusalem',
 'Wellcome Collection']

## 2 · Match by hand first (the skill before the code)

Before automating, do it once by hand so you understand the judgement involved.

1. Go to <https://www.wikidata.org> and search one institution from the list above.
2. Often several items match. **Disambiguate** using the item's **label**, **description**, and
 **statements** (type, location) plus what you already know from your data.
3. Decide: **matched** (confident), **uncertain** (a plausible but unsure match), or **not found**.

> **Prompt you could use (then check it):** *Given an institution name, search Wikidata, return the
> best-matching item's Q-ID, label and description, and flag if it does not look like a museum, library,
> university or archive so I can review it.*

The functions below do exactly this, and record a **match_status**, being honest about uncertainty is
part of the scholarship, not a failure.


In [17]:
import time

WD_API = "https://www.wikidata.org/w/api.php"
# Wikidata asks every client to identify itself; a request without a User-Agent is often
# rejected with HTTP 403. Put your own contact in here if you like.
HEADERS = {"User-Agent": "OpenResearchDH-course/1.0 (teaching example)"}

INSTITUTION_HINTS = ("museum", "library", "gallery", "university", "archive",
                     "collection", "institute", "foundation")

def wd_get(params, tries=5):
    """GET from the Wikidata API, retrying politely on 429 (rate limit) with backoff."""
    for attempt in range(tries):
        r = requests.get(WD_API, headers=HEADERS, params=params, timeout=30)
        if r.status_code == 429:                     # too many requests: wait and retry
            wait = int(r.headers.get("Retry-After", 2 ** attempt))   # seconds
            print(f"    rate-limited, waiting {min(wait, 30)}s...")
            time.sleep(min(wait, 30))
            continue
        r.raise_for_status()
        return r.json()
    r.raise_for_status()                             # retries exhausted: raise the 429
    return r.json()

def wd_search(term, lang=LANG, limit=5):
    """Search Wikidata; return a list of candidate dicts {id,label,description}."""
    data = wd_get({"action": "wbsearchentities", "search": term, "language": lang,
                   "uselang": lang, "format": "json", "limit": limit})
    return data.get("search", [])

def wd_country(qid, lang=LANG):
    """Return the label of the country (P17) of a Wikidata item, or None."""
    data = wd_get({"action": "wbgetentities", "ids": qid, "props": "claims", "format": "json"})
    claims = data["entities"][qid].get("claims", {})
    p17 = claims.get("P17")
    if not p17:
        return None
    country_qid = p17[0]["mainsnak"]["datavalue"]["value"]["id"]
    data2 = wd_get({"action": "wbgetentities", "ids": country_qid, "props": "labels",
                    "languages": lang, "format": "json"})
    return data2["entities"][country_qid]["labels"].get(lang, {}).get("value")

def reconcile_one(term):
    """Return a dict with qid, labels, country, match_status, and (if it failed) the reason."""
    result = {"qid": None, "wikidata_label": None, "wikidata_description": None,
              "country": None, "match_status": None, "error": None}
    try:
        hits = wd_search(term)
    except requests.exceptions.ConnectionError:
        result["match_status"] = "offline"; result["error"] = "no connection to Wikidata"; return result
    except requests.exceptions.Timeout:
        result["match_status"] = "error"; result["error"] = "request timed out"; return result
    except Exception as e:
        result["match_status"] = "error"; result["error"] = f"{type(e).__name__}: {e}"; return result
    if not hits:
        result["match_status"] = "not_found"; return result
    top = hits[0]
    desc = (top.get("description") or "").lower()
    result["qid"] = top["id"]
    result["wikidata_label"] = top.get("label")
    result["wikidata_description"] = top.get("description")
    result["match_status"] = "matched" if any(h in desc for h in INSTITUTION_HINTS) else "uncertain"
    try:
        result["country"] = wd_country(top["id"])
    except Exception:
        result["country"] = None
    return result

## 3 · Reconcile the unique entities (test small, then all)

We reconcile each **unique** institution once (fast), building a reusable mapping table. Inspect it before
merging, check the labels/descriptions look right, and review anything marked *uncertain*.


In [18]:
records = []
for name in entities:
    info = reconcile_one(name)
    info["institution"] = name
    records.append(info)
    reason = f'  ({info["error"]})' if info.get("error") else ""
    print(f'  {name!r:45s} -> {info["match_status"]:9s} {info.get("qid") or ""} {info.get("country") or ""}{reason}')
    time.sleep(1)   # be polite to the free API: pause ~1s between entities to avoid rate limits

mapping = pd.DataFrame(records)[["institution", "qid", "wikidata_label",
                                 "wikidata_description", "country", "match_status"]]
print("\nStatus counts:\n", mapping["match_status"].value_counts())

errs = [r["error"] for r in records if r.get("error")]
if errs:
    print("\nFirst error reported:", errs[0])
mapping

  'Austrian Gallery Belvedere'                  -> not_found  
  'Catholic University of Leuven'               -> matched   Q833670 Belgium
  'Digital Library for Dutch Literature'        -> not_found  
  'Digital Memory of Catalonia'                 -> not_found  
  'Finnish National Gallery'                    -> matched   Q2983474 Finland
  'International Institute of Social History'   -> matched   Q1667757 Netherlands
    rate-limited, waiting 30s...
    rate-limited, waiting 23s...
  'Nationalmuseum Sweden'                       -> matched   Q842858 Sweden
  'Swedish Air Force Museum'                    -> matched   Q1434569 Sweden
  'The Israel Museum, Jerusalem'                -> matched   Q46815 Israel
  'Wellcome Collection'                         -> matched   Q7981191 United Kingdom

Status counts:
 match_status
matched      7
not_found    3
Name: count, dtype: int64


,institution,qid,wikidata_label,wikidata_description,country,match_status
0,Austrian Gallery Belvedere,None,None,None,None,not_found
1,Catholic University of Leuven,Q833670,Katholieke Universiteit Leuven,University since 1968 with its main campus in ...,Belgium,matched
2,Digital Library for Dutch Literature,None,None,None,None,not_found
3,Digital Memory of Catalonia,None,None,None,None,not_found
4,Finnish National Gallery,Q2983474,Finnish National Gallery,organization of three museums that make up the...,Finland,matched
5,International Institute of Social History,Q1667757,International Institute of Social History,historical research institute in Amsterdam,Netherlands,matched
6,Nationalmuseum Sweden,Q842858,Nationalmuseum,"art museum in Stockholm, Sweden",Sweden,matched
7,Swedish Air Force Museum,Q1434569,Swedish Air Force Museum,Swedish military museum,Sweden,matched
8,"The Israel Museum, Jerusalem",Q46815,Israel Museum,national museum of the state of Israel in Jeru...,Israel,matched
9,Wellcome Collection,Q7981191,Wellcome Collection,"museum and library in London, United Kingdom",United Kingdom,matched


### If everything says `offline`
You have no connection to Wikidata right now. The code is correct, run this notebook again on a machine
with internet and it will fill in real Q-IDs and countries. Nothing else needs to change.


## 4 · Save the mapping and merge it back

We save the reusable **entity > Wikidata** map, then join it onto every row of the dataset.


In [19]:
mapping.to_csv(datasets_dir() / "institution_wikidata_map.csv", index=False)

enhanced = df.merge(mapping.rename(columns={"country": "wikidata_country"}),
                    on="institution", how="left")
enhanced.to_csv(datasets_dir() / "van_gogh_enhanced.csv", index=False)
print("Saved: institution_wikidata_map.csv and van_gogh_enhanced.csv")
enhanced.head()

Saved: institution_wikidata_map.csv and van_gogh_enhanced.csv


,id,title,type,institution,country,year,matched_variant,qid,wikidata_label,wikidata_description,wikidata_country,match_status
0,/2024903/photography_ProvidedCHO_KU_Leuven_999...,Vincent van Gogh. Self portrait as painter,IMAGE,Catholic University of Leuven,Belgium,1888.0,Vincent van Gogh,Q833670,Katholieke Universiteit Leuven,University since 1968 with its main campus in ...,Belgium,matched
1,/2024903/photography_ProvidedCHO_KU_Leuven_999...,Vincent van Gogh. Korenveld met cypressen,IMAGE,Catholic University of Leuven,Belgium,1889.0,Vincent van Gogh,Q833670,Katholieke Universiteit Leuven,University since 1968 with its main campus in ...,Belgium,matched
2,/2024903/photography_ProvidedCHO_KU_Leuven_999...,Vincent van Gogh. Olijfgaard,IMAGE,Catholic University of Leuven,Belgium,1889.0,Vincent van Gogh,Q833670,Katholieke Universiteit Leuven,University since 1968 with its main campus in ...,Belgium,matched
3,/2024903/photography_ProvidedCHO_KU_Leuven_999...,Vincent van Gogh. Treurende oude man,IMAGE,Catholic University of Leuven,Belgium,1890.0,Vincent van Gogh,Q833670,Katholieke Universiteit Leuven,University since 1968 with its main campus in ...,Belgium,matched
4,/2024903/photography_ProvidedCHO_KU_Leuven_999...,Vincent van Gogh. Landschap in de Provence bij...,IMAGE,Catholic University of Leuven,Belgium,1890.0,Vincent van Gogh,Q833670,Katholieke Universiteit Leuven,University since 1968 with its main campus in ...,Belgium,matched


## 5 · What the enhancement delivers

- A **Q-ID** column means you can now group by the *entity*, not the *spelling*, the name-variation
 problem from Module 2 is solved at the identifier level.
- A Wikidata **country** lets you cross-check the platform's own country field, or aggregate by country
 reliably.
- You can extend this: swap `P17` for **coordinates** (`P625`) to map institutions, **inception**
 (`P571`) for founding dates, or reconcile a *person* column and pull **birth year**, the pattern is the
 same (search > disambiguate > pull a feature > record match_status).

> **Prompt to extend it:** *Modify `wd_country` to fetch coordinate location (P625) instead of country
> (P17), returning latitude and longitude; keep the same error handling and match_status logic.*


## 6 · Document the step

**Workflow note (for your README):** *Reconciled the `institution` column to Wikidata Q-IDs (search +
disambiguation by description; match_status recorded), added each institution's country (P17), and merged
the result into `van_gogh_enhanced.csv`. Entities marked `uncertain` were kept and flagged for review.*

Record: which column you enriched, which feature you added, how many fell into each match_status, and any
problems.


## 7 · Your turn
1. **Review uncertains:** open the mapping, check any `uncertain`/`not_found` rows, and correct Q-IDs by hand.
2. **A different entity:** set `ENTITY_COLUMN` to a person or place column in your own data.
3. **A different feature:** follow the prompt in Section 5 to pull coordinates or dates instead of country.
4. **Document your decisions**, as always.
